# KGSS 변수 확인 노트북

이 노트북은 KGSS 2003-2025 누적 자료에서 성공 인식 분석에 사용할 주요 변수가 존재하는지 확인하기 위한 스타터 노트북입니다.

원자료(`.sav`)는 Git에 포함하지 않으며, 이 노트북은 로컬 환경에서 `data/raw/kor_data_CUM0074_V2.sav` 파일이 있을 때만 실행됩니다.

## 1. 필요한 라이브러리 불러오기

데이터 처리에는 `pandas`와 `numpy`, SPSS `.sav` 파일 읽기에는 `pyreadstat`, 간단한 시각화 준비를 위해 `matplotlib.pyplot`을 사용합니다.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import pyreadstat
import matplotlib.pyplot as plt

## 2. KGSS 원자료 파일 경로 지정하기

노트북은 `notebooks` 폴더 안에 있으므로, 원자료는 상대 경로 `../data/raw/kor_data_CUM0074_V2.sav`에서 찾습니다. 원자료는 Git에 올리지 않는 파일이므로, 로컬 컴퓨터에 파일이 있을 때만 다음 단계가 실행됩니다.

In [ ]:
data_path = Path("../data/raw/kor_data_CUM0074_V2.sav")

if not data_path.exists():
    raise FileNotFoundError(
        f"KGSS raw data file was not found at: {data_path}\n"
        "Place the local .sav file in data/raw and run this notebook again."
    )

data_path

## 3. SPSS `.sav` 파일 읽기

`pyreadstat.read_sav()`를 사용해 KGSS 원자료를 불러옵니다. `df`에는 데이터가, `meta`에는 변수 라벨 등 메타데이터가 저장됩니다.

In [ ]:
df, meta = pyreadstat.read_sav(data_path)

print("Data loaded successfully.")

## 4. 데이터의 기본 구조 확인하기

데이터의 행과 열 개수, 전체 변수 수, 처음 다섯 행을 확인합니다. 이 단계는 파일이 정상적으로 읽혔는지 빠르게 점검하는 용도입니다.

In [ ]:
print("Data shape:", df.shape)
print("Number of columns:", len(df.columns))

df.head()

## 5. 분석 후보 변수 목록 만들기

분석에 사용할 후보 변수를 목록으로 정리합니다. 성공 인식 변수와 기본 분석에 필요한 연도, 나이, 가중치 변수가 포함되어 있습니다.

In [ ]:
variables_to_check = [
    "YEAR",
    "AGE",
    "FINALWT",
    "SUCDEFRT",
    "SUCDWLTH",
    "SUCDPAED",
    "SUCDKNOW",
    "KIDSOL06",
]

success_variables = [
    "SUCDEFRT",
    "SUCDWLTH",
    "SUCDPAED",
    "SUCDKNOW",
]

child_success_variable = "KIDSOL06"

## 6. 변수 존재 여부 확인하기

후보 변수가 KGSS 데이터 안에 실제로 있는지 확인합니다. 변수명이 다르거나 해당 연도에 포함되지 않은 변수는 `exists` 값이 `False`로 표시됩니다.

In [ ]:
variable_check = pd.DataFrame(
    {
        "variable": variables_to_check,
        "exists": [variable in df.columns for variable in variables_to_check],
    }
)

variable_check

## 7. 각 변수의 값 분포 확인하기

각 후보 변수에 대해 값별 빈도를 출력합니다. 결측값도 함께 확인하기 위해 `dropna=False` 옵션을 사용합니다.

In [ ]:
for variable in variables_to_check:
    print("=" * 80)
    print(f"Value counts for {variable}")
    print("=" * 80)

    if variable not in df.columns:
        print(f"{variable} does not exist in the dataset.\n")
        continue

    print(df[variable].value_counts(dropna=False).sort_index())
    print()

## 8. 성공 인식 변수의 연도별 응답 수 확인하기

`YEAR`를 기준으로 성공 인식 변수들의 결측이 아닌 응답 수를 계산합니다. 이를 통해 어떤 연도에 어떤 성공 인식 문항이 포함되었는지 확인할 수 있습니다.

In [ ]:
if "YEAR" not in df.columns:
    print("YEAR variable does not exist in the dataset.")
else:
    available_success_variables = [
        variable for variable in success_variables if variable in df.columns
    ]

    if not available_success_variables:
        print("None of the success perception variables exist in the dataset.")
    else:
        yearly_success_counts = (
            df.groupby("YEAR")[available_success_variables]
            .count()
            .sort_index()
        )
        display(yearly_success_counts)

## 9. `KIDSOL06` 변수의 연도별 응답 수 확인하기

자녀 관련 성공 기대 문항으로 사용할 수 있는 `KIDSOL06` 변수의 연도별 결측이 아닌 응답 수를 확인합니다.

In [ ]:
if "YEAR" not in df.columns:
    print("YEAR variable does not exist in the dataset.")
elif child_success_variable not in df.columns:
    print(f"{child_success_variable} does not exist in the dataset.")
else:
    yearly_child_success_counts = (
        df.groupby("YEAR")[child_success_variable]
        .count()
        .sort_index()
        .rename("non_null_count")
        .to_frame()
    )
    display(yearly_child_success_counts)

## 10. 다음 단계 메모

이 노트북에서 변수 존재 여부와 연도별 응답 수를 확인한 뒤, 실제 분석 노트북에서는 사용할 연도와 변수 범위를 정하고 가중치(`FINALWT`) 적용 방식, 결측값 처리 방식, 집계표 생성 방식을 결정하면 됩니다.